In [1]:
%cd ..

/Users/philipphager/Documents/GitHub/clix


In [2]:
import altair as alt
import pandas as pd

In [3]:
@alt.theme.register("latex", enable=True)
def theme():
    return {
        "config": {
            "title": {
                "font": "serif",
                "fontWeight": "normal",
                "fontSize": 16,
            },
            "axis": {
                "titleFont": "serif",
                "titleFontWeight": "normal",
                "titleFontSize": 16,
                "labelFont": "serif",
                "labelFontWeight": "normal",
                "labelFontSize": 16,
            },
            "headerColumn": {
                "titleFont": "serif",
                "titleFontWeight": "normal",
                "titleFontSize": 16,
                "labelFont": "serif",
                "labelFontWeight": "normal",
                "labelFontSize": 16,
            },
            "headerRow": {
                "titleFont": "serif",
                "titleFontWeight": "normal",
                "titleFontSize": 16,
                "labelFont": "serif",
                "labelFontWeight": "normal",
                "labelFontSize": 16,
            },
            "legend": {
                "titleFont": "serif",
                "titleFontWeight": "normal",
                "titleFontSize": 16,
                "labelFont": "serif",
                "labelFontWeight": "normal",
                "labelFontSize": 16,
            },
            "text": {
                "font": "serif",
                "fontSize": 14,
            },
        },
    }

In [4]:
df = pd.read_csv("results/2-yandex-embeddings/test.csv")
df.model = df.model.str.upper()

full_df = df[df.embedding == "full"].copy()
hash_df = df[df.embedding == "hash"].copy()
hash_df = pd.concat([hash_df, full_df])
hash_df["embedding"] = "Hashing Trick"

qr_df = df[df.embedding == "qr"].copy()
qr_df = pd.concat([qr_df, full_df])
qr_df["embedding"] = "Quotient-Remainder"

df = pd.concat([hash_df, qr_df])
df["compression_rate"] = df["compression_rate"].map(lambda x: "Full" if x == "full" else x)
df["train_time_min"] = df["train_time_s"] / 60

df.sort_values("model").head()

,model,test_loss,test_ll,test_ppl,test_cond_ppl,train_time_s,embedding,compression_rate,random_state,train_time_min
23,CCM,0.263334,-0.263334,1.338100,1.315977,695.302059,Hashing Trick,Full,3,11.588368
285,CCM,0.270185,-0.270175,1.349902,1.325828,220.184833,Quotient-Remainder,100,1,3.669747
63,CCM,0.267723,-0.267713,1.345219,1.322159,232.168236,Hashing Trick,5,1,3.869471
275,CCM,0.264789,-0.264789,1.340210,1.318073,219.947955,Quotient-Remainder,10,3,3.665799
265,CCM,0.267900,-0.267894,1.344670,1.322082,217.665236,Quotient-Remainder,10,2,3.627754


In [5]:
sort_by = df[df.compression_rate=="Full"].groupby(["model"])["test_ppl"].mean().reset_index()
sort_by = sort_by.sort_values(["test_ppl"])["model"].to_list()

In [6]:
sort_by = ['UBM', 'PBM', 'DBN', 'SDBN', 'CM', 'CCM', 'DCM', 'DCTR', 'DRCTR', 'RCTR', 'GCTR']

In [7]:
filtered_df = df[df.embedding == "Hashing Trick"]

chart_config = {
    "width": 80,
    "height": 190,
    "title": ""
}

def create_chart(y_field, y_title):
    """Create a bar chart with specified y-field and title."""
    return alt.Chart(filtered_df, **chart_config).mark_bar().encode(
        column=alt.Column(
            "compression_rate", 
            title="", 
            sort=["Full", "2", "5", "10", "100", "1000"], 
            spacing=5, 
            header=alt.Header(titleOrient='bottom', labelOrient='bottom')
        ),
        x=alt.X("model", sort="y", title="").axis(labels=False, ticks=False),
        y=alt.Y(y_field, title=y_title).scale(domain=(1.3, 1.5), clamp=True),
        color=alt.Color("model:N", title="", sort=sort_by).scale(range=["#3182bd", "#6baed6", "#31a354", "#a1d99b", "#fd8d3c", "#fdae6b", "#fdd0a2", "#756bb1", "#bcbddc", "#969696", "#bdbdbd"]),
    ).resolve_scale(x="independent", y="shared")

cond_ppl = create_chart("mean(test_cond_ppl)", "Conditional PPL")
ppl = create_chart("mean(test_ppl)", "PPL")

chart = (cond_ppl | ppl).configure_concat(spacing=10)
chart

alt.HConcatChart(...)

In [8]:
import altair as alt

# Chart configuration
chart_config = {
    "width": 80,
    "height": 190,
    "title": ""
}

def create_metric_chart(filtered_df, y_field, y_title, sort_by):
    """Create a bar chart with specified y-field and title."""
    return alt.Chart(filtered_df, **chart_config).mark_bar().encode(
        column=alt.Column(
            "compression_rate", 
            title="", 
            sort=["Full", "2", "5", "10", "100", "1000"], 
            spacing=5, 
            header=alt.Header(titleOrient='bottom', labelOrient='bottom')
        ),
        x=alt.X("model", sort="y", title="").axis(labels=False, ticks=False),
        y=alt.Y(y_field, title=y_title).scale(domain=(1.3, 1.5), clamp=True),
        color=alt.Color("model:N", title="", sort=sort_by).scale(
            range=["#3182bd", "#6baed6", "#31a354", "#a1d99b", "#fd8d3c", 
                   "#fdae6b", "#fdd0a2", "#756bb1", "#bcbddc", "#969696", "#bdbdbd"]
        ),
    ).resolve_scale(x="independent", y="shared")

def create_embedding_charts(df, embedding_type, sort_by):
    """Create conditional PPL and PPL charts for a specific embedding type."""
    filtered_df = df[df.embedding == embedding_type]
    
    cond_ppl = create_metric_chart(filtered_df, "mean(test_cond_ppl)", "Conditional PPL", sort_by)
    ppl = create_metric_chart(filtered_df, "mean(test_ppl)", "PPL", sort_by)
    
    # Concatenate horizontally and add title (no configure here)
    combined_chart = (cond_ppl | ppl)
    return combined_chart.properties(title=f"{embedding_type} Embeddings")

# Create charts for both embedding types
hashing_trick_charts = create_embedding_charts(df, "Hashing Trick", sort_by)
quotient_remainder_charts = create_embedding_charts(df, "Quotient-Remainder", sort_by)

# Combine vertically using vconcat and apply configuration only to final chart
final_chart = (hashing_trick_charts & quotient_remainder_charts).configure_concat(spacing=20)
final_chart

alt.VConcatChart(...)

In [9]:
import pandas as pd
from scipy.stats import spearmanr
from itertools import combinations

import pandas as pd
from scipy.stats import kendalltau
from itertools import combinations

def compute_rank_correlation(df):    
    results = []
    
    for metric in ["test_ppl", "test_cond_ppl"]:
        for embedding in df['embedding'].unique():
            for random_state in df['random_state'].unique():
            
                embedding_data = df[(df['embedding'] == embedding) & (df['random_state'] == random_state)]
                
                # Create ranking matrix for this embedding
                rank_matrix = embedding_data.pivot(index="model", columns="compression_rate", values=metric)
                rank_matrix = rank_matrix.rank(ascending=True)
        
                # Remove models not in all compression rates
                rank_matrix = rank_matrix.dropna()
                
                for rate in sorted(rank_matrix.columns):
                    if rate != "Full":
                        corr, _ = kendalltau(rank_matrix["Full"], rank_matrix[rate])
                        results.append({"embedding": embedding, "random_state": random_state, "metric": metric, "compression_rate": rate, "correlation": corr})
    
    return pd.DataFrame(results)

#correlation_df = compute_rank_correlation(df[~df.model.isin(["GCTR", "RCTR"])])
correlation_df = compute_rank_correlation(df)

(alt.Chart(correlation_df, width=200).mark_line(point=True).encode(
    x=alt.X("compression_rate", title="Compression Ratio", sort=["2", "5", "10", "100", "1000"]).axis(labelAngle=0),
    y=alt.Y("mean(correlation)", title="Kendall's τ"),
    color=alt.Color("metric"),
) + alt.Chart(correlation_df, width=200).mark_errorband(extent="ci").encode(
    x=alt.X("compression_rate", title="Compression Ratio", sort=["2", "5", "10", "100", "1000"]).axis(labelAngle=0),
    y=alt.Y("correlation", title="Kendall's τ"),
    color=alt.Color("metric"),
)).facet(column=alt.Column("embedding", title=""))

alt.FacetChart(...)

In [15]:
correlation_df[correlation_df["compression_rate"] == "10"]

,embedding,random_state,metric,compression_rate,correlation
0,Hashing Trick,1,test_ppl,10,0.955556
5,Hashing Trick,2,test_ppl,10,0.911111
10,Hashing Trick,3,test_ppl,10,0.955556
15,Quotient-Remainder,1,test_ppl,10,1.000000
20,Quotient-Remainder,2,test_ppl,10,0.911111
25,Quotient-Remainder,3,test_ppl,10,0.911111
30,Hashing Trick,1,test_cond_ppl,10,1.000000
35,Hashing Trick,2,test_cond_ppl,10,1.000000
40,Hashing Trick,3,test_cond_ppl,10,1.000000
45,Quotient-Remainder,1,test_cond_ppl,10,1.000000


In [17]:
correlation_df[correlation_df["compression_rate"] == "100"].groupby(["embedding", "metric"])["correlation"].mean()

embedding           metric       
Hashing Trick       test_cond_ppl    0.970370
                    test_ppl         0.896296
Quotient-Remainder  test_cond_ppl    0.985185
                    test_ppl         0.911111
Name: correlation, dtype: float64

In [11]:
alt.Chart(df, width=200).mark_bar(point=True).encode(
    column=alt.Column("embedding:N"),
    x=alt.X("compression_rate", title="Compression Ratio", sort=["Full", "2", "5", "10", "100", "1000"]).axis(labelAngle=0),
    y=alt.Y("mean(train_time_s)"),
)

alt.Chart(...)